# Notebook 3 - Divisão de Treino, Teste e OOT

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path

In [2]:
pasta_notebooks = Path.cwd() if Path.cwd().name == 'notebooks' else Path.cwd() / 'notebooks'
caminho_base_tratada = pasta_notebooks / 'dados' / '2-base_features.pkl'
base_features = pd.read_pickle(caminho_base_tratada)

print(f'Dimensoes: {base_features.shape[0]} linhas x {base_features.shape[1]} colunas')

Dimensoes: 758 linhas x 30 colunas


Criaremos a variável 'flag_safra' para denotar se a observação faz parte do Desenvolvimento ou OOT. Dado o comportamento safra a safra, escolheremos o último mês como OOT.

In [3]:
# Identificacao da safra OOT e das safras de desenvolvimento.
safra_normalizada = (
    base_features['safra']
    .astype(str)
    .str.replace(r'\D', '', regex=True)
)
base_features['flag_safra'] = np.where(
    safra_normalizada.eq('202606'),
    'OOT',
    'DES'
)

print('Distribuicao da flag_safra:')
print(base_features['flag_safra'].value_counts().sort_index())


Distribuicao da flag_safra:
DES    627
OOT    131
Name: flag_safra, dtype: int64


Abaixo, criaremos um dicionário que irá guardar todas as bases a serem utilizadas na modelagem.

In [4]:
bases = dict()
bases['base_oot'] = base_features[base_features.flag_safra=='OOT']

base_features = base_features[base_features.flag_safra=='DES']

Faremos, com observações DES, a divisão usual de treino e teste, na proporção 70/30;.

In [5]:
# Divisao reprodutivel das observacoes DES em treino e teste.
base_treino = base_features.sample(frac=0.7, random_state=42)
base_treino['flag_treino'] = 1

base_teste = base_features.drop(index=base_treino.index)
base_teste['flag_treino'] = 0

bases['base_treino'] = base_treino.reset_index(drop=True)
bases['base_teste'] = base_teste.reset_index(drop=True)

print('Dimensoes das bases:')
for nome, base in bases.items():
    print(f'{nome}: {base.shape[0]} linhas x {base.shape[1]} colunas')


Dimensoes das bases:
base_oot: 131 linhas x 31 colunas
base_treino: 439 linhas x 32 colunas
base_teste: 188 linhas x 32 colunas


In [6]:
# Persistencia de todas as bases utilizadas na modelagem.
for nome, base in bases.items():
    caminho_base = pasta_notebooks / 'dados' / f'3-{nome}.pkl'
    base.to_pickle(caminho_base)
    print(f'{nome}: {caminho_base} ({base.shape[0]} linhas x {base.shape[1]} colunas)')


base_oot: c:\Users\Victor Dogo\Documents\ds3_mlai_test\notebooks\dados\3-base_oot.pkl (131 linhas x 31 colunas)
base_treino: c:\Users\Victor Dogo\Documents\ds3_mlai_test\notebooks\dados\3-base_treino.pkl (439 linhas x 32 colunas)
base_teste: c:\Users\Victor Dogo\Documents\ds3_mlai_test\notebooks\dados\3-base_teste.pkl (188 linhas x 32 colunas)
